In [15]:
import sys
import re
from pathlib import Path
from collections import defaultdict
from bs4 import BeautifulSoup

# Add utils to path
sys.path.append(str(Path.cwd().parent / 'llm_based_annotation'))
from utils_extraction.html_utils import is_manual_label_tag
from utils_extraction.htmlLabel import HTMLLabel

In [16]:
# Define the three HTML files to analyze
from pathlib import Path
data_dir = Path.cwd().parent / 'data' / 'Documents_Annotés'

html_files = [
    data_dir / 'EG' / '1997CanLII16226_ONCA_annotated_EG_tech.html',
    data_dir / 'EG' / '2021QCCA1675_annotated_EG_tech.html',
    data_dir / 'GL' / '1989CanLII1415CITT_annotated_GL.html'
]

# Verify files exist
for f in html_files:
    print(f"{'✓' if f.exists() else '✗'} {f.name}")

✓ 1997CanLII16226_ONCA_annotated_EG_tech.html
✓ 2021QCCA1675_annotated_EG_tech.html
✓ 1989CanLII1415CITT_annotated_GL.html


In [25]:
def extract_parent_level_annotations(html_content):
    """
    Extract all parent-level manual_label annotations (where parent="").
    Returns a dict with keys: 'decision', 'legislation', 'secondary sources'
    Each value is a list of annotation dictionaries containing:
      - full_html: the complete annotation HTML
      - docid: document identifier
      - uri: resource URI
      - text_content: extracted text (no HTML tags)
      - sublabels: list of sublabel types found
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Find all manual_label tags with parent=""
    parent_labels = soup.find_all('manual_label', attrs={'parent': ''})
    
    annotations = {
        'decision': [],
        'legislation': [],
        'secondary sources': []
    }
    
    for label in parent_labels:
        labelname = label.get('labelname', '')
        
        if labelname not in annotations:
            continue
        
        # Extract sublabels
        sublabels = []
        for sublabel in label.find_all('manual_label', recursive=False):
            sublabel_name = sublabel.get('labelname', '')
            sublabels.append(sublabel_name)
        
        # Recursively get all sublabels (nested)
        all_sublabels = [sl.get('labelname', '') for sl in label.find_all('manual_label')]
        
        annotation_data = {
            'full_html': str(label),
            'docid': label.get('docid', ''),
            'uri': label.get('uri', ''),
            'text_content': label.get_text(strip=True),
            'direct_sublabels': sublabels,
            'all_sublabels': all_sublabels
        }
        
        annotations[labelname].append(annotation_data)
    
    return annotations

In [26]:
# Process all files
all_annotations = {
    'decision': [],
    'legislation': [],
    'secondary sources': []
}

for html_file in html_files:
    print(f"\nProcessing: {html_file.name}")
    
    with open(html_file, 'r', encoding='utf-8') as f:
        html_content = f.read()
    
    annotations = extract_parent_level_annotations(html_content)
    
    # Aggregate results
    for label_type in ['decision', 'legislation', 'secondary sources']:
        count = len(annotations[label_type])
        print(f"  - {label_type}: {count} annotations")
        all_annotations[label_type].extend(annotations[label_type])

print("\n" + "="*60)
print("TOTAL ANNOTATIONS ACROSS ALL FILES:")
for label_type in ['decision', 'legislation', 'secondary sources']:
    print(f"  {label_type}: {len(all_annotations[label_type])}")
print("="*60)


Processing: 1997CanLII16226_ONCA_annotated_EG_tech.html
  - decision: 248 annotations
  - legislation: 355 annotations
  - secondary sources: 15 annotations

Processing: 2021QCCA1675_annotated_EG_tech.html
  - decision: 53 annotations
  - legislation: 45 annotations
  - secondary sources: 1 annotations

Processing: 1989CanLII1415CITT_annotated_GL.html
  - decision: 17 annotations
  - legislation: 34 annotations
  - secondary sources: 33 annotations

TOTAL ANNOTATIONS ACROSS ALL FILES:
  decision: 318
  legislation: 434
  secondary sources: 49


In [27]:
print(f"DECISION ANNOTATIONS ({len(all_annotations['decision'])} total)\\n")
print("="*80)

for i, annotation in enumerate(all_annotations['decision'][:20], 1):  # Show first 20
    print(f"\n[{i}] DocID: {annotation['docid']}")
    print(f"    URI: {annotation['uri']}")
    print(f"    Sublabels: {', '.join(set(annotation['all_sublabels']))}")
    print(f"    Text: {annotation['text_content'][:100]}...")
    print(f"    HTML Preview: {annotation['full_html'][:200]}...")

if len(all_annotations['decision']) > 20:
    print(f"\n... and {len(all_annotations['decision']) - 20} more decision annotations")

DECISION ANNOTATIONS (318 total)\n

[1] DocID: Church of Scientology ONCA 1997
    URI: https://canlii.ca/t/6hxv
    Sublabels: title
    Text: R. v. Church of Scientology...
    HTML Preview: <manual_label docid="Church of Scientology ONCA 1997" labelname="decision" parent="" style="background-color: rgb(106, 163, 255); color: black;" uri="https://canlii.ca/t/6hxv" verified="false"><manual...

[2] DocID: Church of Scientology ONCA 1997
    URI: https://canlii.ca/t/6hxv
    Sublabels: citation
    Text: 33 O.R. (3d) 65...
    HTML Preview: <manual_label docid="Church of Scientology ONCA 1997" labelname="decision" parent="" style="background-color: rgb(106, 163, 255); color: black;" uri="https://canlii.ca/t/6hxv" verified="false"><manual...

[3] DocID: Church of Scientology ONCA 1997
    URI: https://canlii.ca/t/6hxv
    Sublabels: citation
    Text: [1997] O.J. No. 1548...
    HTML Preview: <manual_label docid="Church of Scientology ONCA 1997" labelname="decision" parent="" style="back

In [28]:
print(f"LEGISLATION ANNOTATIONS ({len(all_annotations['legislation'])} total)\\n")
print("="*80)

for i, annotation in enumerate(all_annotations['legislation'][:20], 1):  # Show first 20
    print(f"\n[{i}] DocID: {annotation['docid']}")
    print(f"    URI: {annotation['uri']}")
    print(f"    Sublabels: {', '.join(set(annotation['all_sublabels']))}")
    print(f"    Text: {annotation['text_content'][:100]}...")
    print(f"    HTML Preview: {annotation['full_html'][:200]}...")

if len(all_annotations['legislation']) > 20:
    print(f"\n... and {len(all_annotations['legislation']) - 20} more legislation annotations")

LEGISLATION ANNOTATIONS (434 total)\n

[1] DocID: Charter
    URI: https://canlii.ca/t/ldsx
    Sublabels: title
    Text: Charter of Rights and Freedoms...
    HTML Preview: <manual_label docid="Charter" labelname="legislation" parent="" style="background-color: rgb(118, 206, 222); color: black;" uri="https://canlii.ca/t/ldsx" verified="false"><manual_label labelname="tit...

[2] DocID: Charter
    URI: https://canlii.ca/t/ldsx
    Sublabels: fragment, title
    Text: s. 8ofCharter...
    HTML Preview: <manual_label docid="Charter" labelname="legislation" parent="" style="background-color: rgb(118, 206, 222); color: black;" uri="https://canlii.ca/t/ldsx" verified="false"><manual_label fragmentid="se...

[3] DocID: Charter
    URI: https://canlii.ca/t/ldsx
    Sublabels: fragment, title
    Text: Canadian Charter of
Rights and Freedoms,ss. 8,24(2)...
    HTML Preview: <manual_label docid="Charter" labelname="legislation" parent="" style="background-color: rgb(118, 206, 222); color: bla

In [29]:
print(f"SECONDARY SOURCE ANNOTATIONS ({len(all_annotations['secondary sources'])} total)\\n")
print("="*80)

if len(all_annotations['secondary sources']) == 0:
    print("No secondary source annotations found in the analyzed files.")
else:
    for i, annotation in enumerate(all_annotations['secondary sources'][:20], 1):
        print(f"\n[{i}] DocID: {annotation['docid']}")
        print(f"    URI: {annotation['uri']}")
        print(f"    Sublabels: {', '.join(set(annotation['all_sublabels']))}")
        print(f"    Text: {annotation['text_content'][:100]}...")
        print(f"    HTML Preview: {annotation['full_html'][:200]}...")
    
    if len(all_annotations['secondary sources']) > 20:
        print(f"\n... and {len(all_annotations['secondary sources']) - 20} more secondary source annotations")

SECONDARY SOURCE ANNOTATIONS (49 total)\n

[1] DocID: Study of the Constitution
    URI: None
    Sublabels: authors, source, title, fragment
    Text: Dicey,Introduction to the Study of the Law of the Constitution,10th ed.
(1959),p. 193...
    HTML Preview: <manual_label docid="Study of the Constitution" labelname="secondary sources" parent="" style="background-color: rgb(39, 143, 227); color: white;" uri="None" verified="false"><manual_label labelname="...

[2] DocID: The Stranger in our Midst
    URI: None
    Sublabels: authors, source, title
    Text: Head, I., "The Stranger in our Midst: A Sketch of the Legal
Status of the Alien in Canada"(1964), Ca...
    HTML Preview: <manual_label docid="The Stranger in our Midst" labelname="secondary sources" parent="" style="background-color: rgb(39, 143, 227); color: white;" uri="None" verified="false"> <manual_label labelname=...

[3] DocID: Reform of the Criminal Jury
    URI: https://canlii.ca/t/2blz
    Sublabels: authors, source, title

In [31]:
from collections import Counter

def analyze_sublabel_patterns(annotations, label_type):
    """Analyze which sublabels appear and in what patterns."""
    
    sublabel_counts = Counter()
    pattern_counts = Counter()
    
    for ann in annotations:
        # Count individual sublabels
        for sublabel in ann['all_sublabels']:
            sublabel_counts[sublabel] += 1
        
        # Count patterns (combinations of sublabels)
        pattern = tuple(sorted(set(ann['all_sublabels'])))
        pattern_counts[pattern] += 1
    
    print(f"\n{label_type.upper()} - SUBLABEL ANALYSIS")
    print("="*60)
    
    print("\nMost common sublabels:")
    for sublabel, count in sublabel_counts.most_common(10):
        print(f"  {sublabel}: {count}")
    
    print("\nMost common sublabel patterns:")
    for pattern, count in pattern_counts.most_common(10):
        print(f"  {pattern}: {count}")
    
    return sublabel_counts, pattern_counts

# Analyze each type
for label_type in ['decision', 'legislation', 'secondary sources']:
    if all_annotations[label_type]:
        analyze_sublabel_patterns(all_annotations[label_type], label_type)


DECISION - SUBLABEL ANALYSIS

Most common sublabels:
  citation: 411
  title: 245
  fragment: 134

Most common sublabel patterns:
  ('citation', 'title'): 100
  ('title',): 96
  ('fragment',): 53
  ('citation', 'fragment', 'title'): 39
  ('citation',): 14
  ('fragment', 'title'): 9
  (): 4
  ('citation', 'fragment'): 3

LEGISLATION - SUBLABEL ANALYSIS

Most common sublabels:
  fragment: 367
  title: 265
  citation: 34

Most common sublabel patterns:
  ('fragment',): 165
  ('fragment', 'title'): 140
  ('title',): 97
  ('citation', 'title'): 15
  ('citation', 'fragment', 'title'): 10
  ('citation',): 6
  ('citation', 'fragment'): 1

SECONDARY SOURCES - SUBLABEL ANALYSIS

Most common sublabels:
  title: 44
  authors: 21
  source: 21
  fragment: 10

Most common sublabel patterns:
  ('title',): 16
  ('source', 'title'): 9
  ('authors', 'fragment', 'source', 'title'): 7
  ('authors', 'title'): 5
  ('authors',): 5
  ('authors', 'source', 'title'): 4
  ('fragment',): 2
  ('fragment', 'source'

In [32]:
# Create clean lists for each category
decision_list = all_annotations['decision']
legislation_list = all_annotations['legislation']
secondary_source_list = all_annotations['secondary sources']

print(f"Created lists:")
print(f"  - decision_list: {len(decision_list)} items")
print(f"  - legislation_list: {len(legislation_list)} items")
print(f"  - secondary_source_list: {len(secondary_source_list)} items")
print("\nThese lists are now available for further analysis in subsequent cells.")

Created lists:
  - decision_list: 318 items
  - legislation_list: 434 items
  - secondary_source_list: 49 items

These lists are now available for further analysis in subsequent cells.


In [35]:
def simplify_html(html_str):
    """
    Simplify HTML by converting <manual_label labelname="X"> to <X>.
    """
    soup = BeautifulSoup(html_str, 'html.parser')
    
    # Find all manual_label tags
    for tag in soup.find_all('manual_label'):
        labelname = tag.get('labelname', '')
        if labelname:
            # Create new tag with just the labelname
            new_tag = soup.new_tag(labelname)
            # Move children to new tag
            for child in list(tag.children):
                new_tag.append(child)
            # Replace old tag with new tag
            tag.replace_with(new_tag)
    
    return str(soup)


def analyze_sublabel_order(annotations, label_type, examples=0):
    """
    Analyze the order in which sublabels appear within each annotation.
    Returns ordered patterns showing which sublabels come before others.
    
    Args:
        annotations: List of annotation dictionaries
        label_type: Type of label ('decision', 'legislation', etc.)
        examples: Number of examples to show per pattern (default: 0 = no examples)
    """
    from collections import Counter, defaultdict
    
    ordered_pattern_counts = Counter()
    pattern_examples = defaultdict(list)  # Store examples for each pattern
    
    for ann in annotations:
        # Parse the HTML to get sublabels in order
        soup = BeautifulSoup(ann['full_html'], 'html.parser')
        
        # Find all manual_label tags in order (depth-first traversal)
        sublabels_in_order = []
        for sublabel in soup.find_all('manual_label'):
            sublabel_name = sublabel.get('labelname', '')
            if sublabel_name:
                sublabels_in_order.append(sublabel_name)
        
        # Create ordered pattern (as tuple to make it hashable)
        if sublabels_in_order:
            ordered_pattern = tuple(sublabels_in_order)
            ordered_pattern_counts[ordered_pattern] += 1
            
            # Store example if we need examples and haven't collected enough yet
            if examples > 0 and len(pattern_examples[ordered_pattern]) < examples:
                pattern_examples[ordered_pattern].append(ann)
    
    print(f"\n{label_type.upper()} - SUBLABEL ORDER ANALYSIS")
    print("="*60)
    print(f"Total annotations: {len(annotations)}")
    print(f"Unique ordered patterns: {len(ordered_pattern_counts)}")
    
    print("\nMost common ordered sublabel patterns:")
    for pattern, count in ordered_pattern_counts.most_common(20):
        # Format with arrows to show order
        pattern_str = " → ".join(pattern)
        print(f"  [{count:3d}x] {pattern_str}")
        
        # Show examples if requested
        if examples > 0 and pattern in pattern_examples:
            num_to_show = min(examples, len(pattern_examples[pattern]))
            print(f"        Examples ({num_to_show}):")
            for i, example in enumerate(pattern_examples[pattern][:num_to_show], 1):
                simplified = simplify_html(example['full_html'])
                # Truncate if too long
                if len(simplified) > 300:
                    simplified = simplified[:300] + "..."
                print(f"        {i}. {simplified}")
            print()
    
    return ordered_pattern_counts

# Analyze order for each type
print("\n" + "="*80)
print("ANALYZING SUBLABEL ORDER (which comes before which)")
print("="*80)

for label_type in ['decision', 'legislation', 'secondary sources']:
    if all_annotations[label_type]:
        ordered_patterns = analyze_sublabel_order(all_annotations[label_type], label_type, examples=1)
        print()  # Extra line between sections


ANALYZING SUBLABEL ORDER (which comes before which)

DECISION - SUBLABEL ORDER ANALYSIS
Total annotations: 318
Unique ordered patterns: 25

Most common ordered sublabel patterns:
  [ 96x] decision → title
        Examples (1):
        1. <decision><title>R. v. Church of Scientology</title></decision>

  [ 43x] decision → fragment
        Examples (1):
        1. <decision><fragment>p. 831</fragment>
S.C.R.</decision>

  [ 31x] decision → title → citation → citation
        Examples (1):
        1. <decision><title>Rhône (The) v. Peter A.B. Widener (The)</title>, <citation>[1993] 1
S.C.R. 497</citation>, <citation>101 D.L.R. (4th) 188</citation></decision>

  [ 27x] decision → title → citation
        Examples (1):
        1. <decision><title>R. v. Church of Scientology (1992)</title>, <citation>9 C.R.R. (2d)
196 (Ont. Gen. Div.)</citation></decision>

  [ 20x] decision → title → citation → fragment
        Examples (1):
        1. <decision><title>Canadian Dredge &amp; Dock Co. v. R.,